In [16]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from scipy.stats import norm

In [2]:
apple = pd.read_parquet('Apple_features.parquet')
btc = pd.read_parquet('BTC_features.parquet')
qqq = pd.read_parquet('QQQ_features.parquet')
spx = pd.read_parquet('spx_features.parquet')
tesla = pd.read_parquet('spx_features.parquet')

In [3]:
apple.head()

,target_3,logret_3,dist_sma3,vol3,volume_ratio3,target_5,logret_5,dist_sma5,vol5,volume_ratio5,...,range_pct,body,close_location,upper_shadow,lower_shadow,dist_sma200,momentum_accel,parkinson_vol,abs_ret_1,streak
Date,,,,,,,,,,,,,,,,,,,,,
2015-09-16,-0.010362,0.019080,0.003535,0.004580,0.803010,-0.018117,0.055275,0.012648,0.007644,0.738488,...,0.009449,0.001376,0.881821,0.001117,0.006958,-0.034260,0.086572,0.005696,0.001117,5.0
2015-09-17,-0.004575,-0.012128,-0.013993,0.015651,1.329886,0.009436,0.011921,-0.011334,0.014244,1.267517,...,0.024315,-0.015044,0.072201,0.007286,0.001756,-0.054928,0.094888,0.014453,0.021622,-1.0
2015-09-18,0.007639,-0.024639,-0.009977,0.011906,1.269318,0.011045,-0.006676,-0.014113,0.012639,1.339555,...,0.021419,0.011051,0.650203,0.007492,0.002997,-0.058837,0.124087,0.012906,0.004134,-2.0
2015-09-21,-0.001825,-0.010362,0.008903,0.018518,0.798781,-0.024337,-0.000867,0.001356,0.014079,0.933028,...,0.014842,0.013548,0.906431,0.001389,0.000087,-0.044266,0.182428,0.008968,0.015395,1.0
2015-09-22,0.011486,-0.004575,-0.005438,0.015778,0.863801,-0.039023,-0.025080,-0.009417,0.014559,0.911608,...,0.014638,0.000176,0.530122,0.006878,0.007584,-0.059256,0.201808,0.008795,0.015835,-1.0


In [4]:
apple.columns

Index(['target_3', 'logret_3', 'dist_sma3', 'vol3', 'volume_ratio3',
       'target_5', 'logret_5', 'dist_sma5', 'vol5', 'volume_ratio5',
       'target_10', 'logret_10', 'dist_sma10', 'vol10', 'volume_ratio10',
       'target_30', 'logret_30', 'dist_sma30', 'vol30', 'volume_ratio30',
       'target_60', 'logret_60', 'dist_sma60', 'vol60', 'volume_ratio60',
       'vol_ratio', 'drawdown', 'range_pct', 'body', 'close_location',
       'upper_shadow', 'lower_shadow', 'dist_sma200', 'momentum_accel',
       'parkinson_vol', 'abs_ret_1', 'streak'],
      dtype='object')

In [14]:
dfs = {
    "apple": apple,
    "btc": btc,
    "qqq": qqq,
    "spx": spx,
    "tesla": tesla,
}

def test(
    df,
    target_col,
    hac_lags=None,
    exclude_targets=True):
    df = df.copy()
    y = df[target_col]
    if exclude_targets:
        X_cols = [c for c in df.columns if not c.startswith("target_")]
    else:
        X_cols = [c for c in df.columns if c != target_col]
    X = df[X_cols]
    data = pd.concat([y, X], axis=1).dropna()
    y = data[target_col]
    X = data[X_cols]
    X = sm.add_constant(X)
    if hac_lags is None:
        try:
            hac_lags = int(target_col.split("_")[1])
        except:
            hac_lags = 5
    model = sm.OLS(y, X).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": hac_lags}
    )
    return model
res = []
for name, df in dfs.items():
    target_cols = [c for c in df.columns if c.startswith("target_")]
    for target_col in target_cols:
        model = test(df, target_col)
        row = {
            "asset": name,
            "target": target_col,
            "nobs": int(model.nobs),
            "alpha": model.params["const"],
            "alpha_pvalue": model.pvalues["const"],
            "joint_beta_pvalue": float(
                model.f_test(
                    " + ".join([f"{c}=0" for c in model.params.index if c != "const"])
                ).pvalue
            ),
            "r2": model.rsquared,
            "adj_r2": model.rsquared_adj,
        }
        for c in model.params.index:
            if c != "const":
                row[f"{c}_beta"] = model.params[c]
                row[f"{c}_pvalue"] = model.pvalues[c]
        results.append(row)
test_results = pd.DataFrame(results)

In [15]:
test_results[
    ["asset", "target", "nobs", "alpha", "alpha_pvalue", 
     "joint_beta_pvalue", "r2", "adj_r2"]
].sort_values(["asset", "target"])

,asset,target,nobs,alpha,alpha_pvalue,joint_beta_pvalue,r2,adj_r2
2,apple,target_10,2631,-0.025708,0.103912,3.670471e-02,0.051537,0.039855
27,apple,target_10,2631,-0.025708,0.103912,3.670471e-02,0.051537,0.039855
0,apple,target_3,2631,-0.011970,0.081664,7.669942e-02,0.035839,0.023963
25,apple,target_3,2631,-0.011970,0.081664,7.669942e-02,0.035839,0.023963
3,apple,target_30,2631,-0.033031,0.336139,5.069324e-06,0.087995,0.076761
28,apple,target_30,2631,-0.033031,0.336139,5.069324e-06,0.087995,0.076761
1,apple,target_5,2631,-0.015300,0.107737,5.937513e-02,0.043548,0.031767
26,apple,target_5,2631,-0.015300,0.107737,5.937513e-02,0.043548,0.031767
4,apple,target_60,2631,-0.041679,0.420325,1.367028e-09,0.128421,0.117686
29,apple,target_60,2631,-0.041679,0.420325,1.367028e-09,0.128421,0.117686


In [17]:
def clark_west(y, X, train_frac=0.7):
    n = len(y)
    split = int(n * train_frac)
    y_train, y_test = y.iloc[:split], y.iloc[split:]
    X_train, X_test = X.iloc[:split], X.iloc[split:]
    model = sm.OLS(y_train, sm.add_constant(X_train)).fit()
    y_model = model.predict(sm.add_constant(X_test))
    y_bench = np.repeat(y_train.mean(), len(y_test))
    e_bench = y_test.values - y_bench
    e_model = y_test.values - y_model
    f = (e_bench**2 - (e_model**2 - (y_bench - y_model)**2))
    cw_reg = sm.OLS(f, np.ones(len(f))).fit(cov_type="HAC", cov_kwds={"maxlags": 5})
    t_stat = cw_reg.tvalues[0]
    p_value = 1 - norm.cdf(t_stat)  
    mse_model = np.mean(e_model**2)
    mse_bench = np.mean(e_bench**2)
    r2_os = 1 - mse_model / mse_bench
    return t_stat, p_value, r2_os
results = []
for asset, df in dfs.items():
    targets = [c for c in df.columns if c.startswith("target_")]
    features = [c for c in df.columns if not c.startswith("target_")]
    for target in targets:
        data = df[[target] + features].dropna()
        y = data[target]
        X = data[features]
        t_stat, p_value, r2_os = clark_west(y, X)
        results.append({
            "asset": asset,
            "target": target,
            "CW_t": t_stat,
            "p_value": p_value,
            "OOS_R2": r2_os})
cw_results = pd.DataFrame(results)
print(cw_results.sort_values(["asset", "target"]))

    asset     target      CW_t   p_value    OOS_R2
2   apple  target_10 -0.104206  0.541497 -0.098253
0   apple   target_3  0.829690  0.203357 -0.040839
3   apple  target_30 -0.505598  0.693430 -0.264260
1   apple   target_5  1.028042  0.151965 -0.033265
4   apple  target_60  1.258120  0.104174 -0.094946
7     btc  target_10 -1.413832  0.921294 -0.136935
5     btc   target_3 -0.551826  0.709466 -0.052064
8     btc  target_30 -1.407908  0.920421 -0.254872
6     btc   target_5 -1.047673  0.852605 -0.084990
9     btc  target_60 -2.486733  0.993554 -0.574841
12    qqq  target_10 -0.443067  0.671141 -0.147079
10    qqq   target_3  0.960754  0.168338 -0.063054
13    qqq  target_30 -1.053410  0.853923 -0.327609
11    qqq   target_5  0.687605  0.245851 -0.059991
14    qqq  target_60 -0.120046  0.547777 -0.168628
17    spx  target_10 -0.263786  0.604028 -0.138998
15    spx   target_3  1.251686  0.105342 -0.076976
18    spx  target_30 -1.038337  0.850443 -0.243581
16    spx   target_5  0.842591 

/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_54665/703189265.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  t_stat = cw_reg.tvalues[0]
/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_54665/703189265.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  t_stat = cw_reg.tvalues[0]
/var/folders/43/y646xgv56z72xq7sm281s4zr0000gn/T/ipykernel_54665/703189265.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
